# 1. Missing Data: Detection, Visualization, and Imputation

Missing data is one of the most common issues in real-world datasets. This notebook covers:
- Detecting and reporting missing values
- Identifying hidden missing values
- Simple imputation (mean, median, mode)
- Advanced imputation (KNN, Iterative/MICE)
- Missing indicators and their predictive value

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

## 1.1 Detecting Missing Values

The first step is always a **missing data report**: for each column, count the number and percentage of missing values.

In [ ]:
url = ("https://raw.githubusercontent.com/datasciencedojo/"
       "datasets/master/titanic.csv")
df = pd.read_csv(url)
print(f"Shape: {df.shape}")

def missing_report(df):
    """Return a DataFrame summarizing missing data."""
    missing = df.isnull().sum()
    pct = (missing / len(df) * 100).round(1)
    report = pd.DataFrame({'missing': missing, 'pct': pct})
    return report[report['missing'] > 0].sort_values('pct', ascending=False)

print(missing_report(df))

## 1.2 Hidden Missing Values

Some datasets encode missing values as special strings (e.g., `" ?"`, `"N/A"`, `"Unknown"`). Always pass these to `na_values` when loading.

In [ ]:
adult_url = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
             "adult/adult.data")
adult_cols = ["age", "workclass", "fnlwgt", "education", "education_num",
              "marital_status", "occupation", "relationship", "race",
              "sex", "capital_gain", "capital_loss", "hours_per_week",
              "native_country", "income"]

df_adult = pd.read_csv(adult_url, header=None, names=adult_cols,
                        na_values=[" ?"], skipinitialspace=True)
print(f"Adult Census shape: {df_adult.shape}")
print(f"\nMissing values (after replacing ' ?'):")
print(missing_report(df_adult))

## 1.3 Simple Imputation

sklearn's `SimpleImputer` offers strategy-based imputation:
- `"median"` for numerical columns (robust to outliers)
- `"most_frequent"` for categorical columns
- Manual `fillna()` with a constant for specific cases

In [ ]:
df_simple = df.copy()

# Median imputation for Age
num_imputer = SimpleImputer(strategy='median')
df_simple['Age'] = num_imputer.fit_transform(df_simple[['Age']])

# Mode imputation for Embarked
cat_imputer = SimpleImputer(strategy='most_frequent')
df_simple['Embarked'] = cat_imputer.fit_transform(
    df_simple[['Embarked']]).ravel()

# Constant for Cabin
df_simple['Cabin'] = df_simple['Cabin'].fillna('Unknown')

print("After simple imputation:")
print(df_simple.isnull().sum())

## 1.4 KNN Imputation

**KNN imputation** estimates missing values using the weighted average of $k$ nearest complete neighbors. It preserves correlations between features better than univariate methods.

In [ ]:
df_knn = df.copy()
num_cols = ['Age', 'Fare', 'SibSp', 'Parch']

knn_imputer = KNNImputer(n_neighbors=5, weights='distance')
df_knn[num_cols] = knn_imputer.fit_transform(df_knn[num_cols])

print(f"Missing after KNN imputation: {df_knn[num_cols].isnull().sum().sum()}")
print(f"\nAge statistics after KNN imputation:")
print(df_knn['Age'].describe())

## 1.5 Iterative Imputation (MICE)

**MICE** (Multiple Imputation by Chained Equations) models each feature with missing values as a function of other features. It iterates through all features with missing data, refining imputations over multiple rounds.

In [ ]:
df_mice = df.copy()
iter_imputer = IterativeImputer(max_iter=10, random_state=42)
df_mice[num_cols] = iter_imputer.fit_transform(df_mice[num_cols])

print(f"Missing after iterative imputation: {df_mice[num_cols].isnull().sum().sum()}")
print(f"\nAge statistics after MICE:")
print(df_mice['Age'].describe())

## 1.6 Comparing Imputation Methods

Each method produces slightly different distributions. Compare their summary statistics to understand the impact.

In [ ]:
comparison = pd.DataFrame({
    'Original': df['Age'].describe(),
    'Median': df_simple['Age'].describe(),
    'KNN': df_knn['Age'].describe(),
    'MICE': df_mice['Age'].describe(),
})
print(comparison.round(2))

## 1.7 Missing Indicators

The **fact that a value is missing** can itself be informative. Adding binary indicators for missingness can improve model performance.

In [ ]:
df_ind = df.copy()
df_ind['Age_was_missing'] = df_ind['Age'].isnull().astype(int)
df_ind['Cabin_was_missing'] = df_ind['Cabin'].isnull().astype(int)

print("Survival rate by Age missingness:")
print(df_ind.groupby('Age_was_missing')['Survived'].mean())
print("\nSurvival rate by Cabin missingness:")
print(df_ind.groupby('Cabin_was_missing')['Survived'].mean())

## Key Takeaways

| Method | Pros | Cons |
|---|---|---|
| Median/Mode | Simple, fast | Ignores feature correlations |
| KNN | Preserves local structure | Slower, sensitive to $k$ |
| MICE | Models feature interactions | Computationally expensive |
| Missing Indicator | Captures informative missingness | Increases dimensionality |